In [1]:
!pip -q install "transformers==4.46.3" "datasets>=2.20" "accelerate>=0.34" sentencepiece scikit-learn 2>/dev/null
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import transformers
print(transformers.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 148.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 124.8 MB/s eta 0:00:00
4.46.3 True NVIDIA A100-SXM4-40GB


In [2]:
BACKBONE = "microsoft/deberta-v3-base"
MAX_LENGTH = 128
INTENT_LABELS = ["similar_items", "graph_pairing", "color_variant", "composite_intent", "chit_chat"]
LABEL2ID = {l: i for i, l in enumerate(INTENT_LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
DATA_DIR = "/content"
OUTPUT_DIR = "/content/intent_classifier_deberta"

SEED = 42
NUM_EPOCHS = 20
EARLY_STOP_PATIENCE = 4
LR = 2e-5
BATCH = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
LABEL_SMOOTHING = 0.1
LLRD_DECAY = 0.9
LR_SCHEDULER = "cosine"

import random, numpy as np, torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [3]:
from google.colab import files
files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


{'test.csv': b'query,intent\r\nblack mini skirt,similar_items\r\npleated tennis skirt,similar_items\r\nwhite denim skirt,similar_items\r\nsatin slip skirt,similar_items\r\ncargo maxi skirt,similar_items\r\nleopard midi skirt,similar_items\r\nlinen wrap skirt,similar_items\r\nribbed knit skirt,similar_items\r\nhigh waisted skirt,similar_items\r\ntulle ballet skirt,similar_items\r\ncorduroy button skirt,similar_items\r\nasymmetrical hem skirt,similar_items\r\nsequined party skirt,similar_items\r\nfaux leather skirt,similar_items\r\nchecked pencil skirt,similar_items\r\ntiered boho skirt,similar_items\r\na line skirt,similar_items\r\nlow rise mini,similar_items\r\nfloral chiffon skirt,similar_items\r\ntailored wool skirt,similar_items\r\ndistressed denim mini,similar_items\r\nmesh overlay skirt,similar_items\r\nruched bodycon skirt,similar_items\r\nolive utility skirt,similar_items\r\nplisse midi skirt,similar_items\r\nblack ankle boots,similar_items\r\nwhite leather sneakers,similar_item

In [4]:
import pandas as pd, numpy as np, torch
from datasets import Dataset

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df   = pd.read_csv(f"{DATA_DIR}/val.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test.csv")
for d, n in [(train_df, "train"), (val_df, "val"), (test_df, "test")]:
    print(n, len(d), dict(d["intent"].value_counts()))

def to_ds(df):
    df = df.copy()
    df["label"] = df["intent"].map(LABEL2ID)
    return Dataset.from_pandas(df[["query", "label"]], preserve_index=False)

ds_train, ds_val, ds_test = to_ds(train_df), to_ds(val_df), to_ds(test_df)

train 4816 {'color_variant': np.int64(1006), 'graph_pairing': np.int64(1004), 'chit_chat': np.int64(965), 'similar_items': np.int64(940), 'composite_intent': np.int64(901)}
val 652 {'color_variant': np.int64(137), 'graph_pairing': np.int64(134), 'chit_chat': np.int64(131), 'similar_items': np.int64(127), 'composite_intent': np.int64(123)}
test 974 {'similar_items': np.int64(200), 'graph_pairing': np.int64(198), 'color_variant': np.int64(196), 'composite_intent': np.int64(191), 'chit_chat': np.int64(189)}


In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

def tok(batch):
    return tokenizer(batch["query"], truncation=True, max_length=MAX_LENGTH)

ds_train = ds_train.map(tok, batched=True)
ds_val   = ds_val.map(tok, batched=True)
ds_test  = ds_test.map(tok, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/4816 [00:00<?, ? examples/s]

Map:   0%|          | 0/652 [00:00<?, ? examples/s]

Map:   0%|          | 0/974 [00:00<?, ? examples/s]

In [6]:
counts = train_df["intent"].map(LABEL2ID).value_counts().sort_index()
weights = (counts.sum() / (len(counts) * counts)).reindex(range(len(INTENT_LABELS))).fillna(1.0)
class_weights = torch.tensor(weights.values, dtype=torch.float)
print(dict(zip(INTENT_LABELS, [round(w, 3) for w in class_weights.tolist()])))

{'similar_items': 1.025, 'graph_pairing': 0.959, 'color_variant': 0.957, 'composite_intent': 1.069, 'chit_chat': 0.998}


In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch.nn as nn

model = AutoModelForSequenceClassification.from_pretrained(
    BACKBONE, num_labels=len(INTENT_LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
)
collator = DataCollatorWithPadding(tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    out = {"f1_macro": f1_score(labels, preds, average="macro"), "accuracy": accuracy_score(labels, preds)}
    per = f1_score(labels, preds, average=None, labels=list(range(len(INTENT_LABELS))))
    for i, lab in enumerate(INTENT_LABELS):
        out[f"f1_{lab}"] = per[i]
    return out

def build_llrd_param_groups(model, base_lr, weight_decay, decay):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    n_layers = model.config.num_hidden_layers
    groups = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if ".encoder.layer." in name:
            layer_i = int(name.split(".encoder.layer.")[1].split(".")[0])
            lr = base_lr * (decay ** (n_layers - layer_i))
        elif "embeddings" in name:
            lr = base_lr * (decay ** (n_layers + 1))
        else:
            lr = base_lr
        wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        groups.append({"params": [p], "lr": lr, "weight_decay": wd})
    return groups

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = class_weights.to(device=logits.device, dtype=logits.dtype)
        loss = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is None:
            try:
                self.optimizer = torch.optim.AdamW(build_llrd_param_groups(self.model, LR, WEIGHT_DECAY, LLRD_DECAY), lr=LR)
            except Exception:
                self.optimizer = super().create_optimizer()
        return self.optimizer

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=64,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    bf16=False,
    fp16=False,
    group_by_length=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)

trainer_kwargs = dict(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)
try:
    trainer = WeightedTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    trainer = WeightedTrainer(tokenizer=tokenizer, **trainer_kwargs)
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Similar Items,F1 Graph Pairing,F1 Color Variant,F1 Composite Intent,F1 Chit Chat
1,0.988200,0.690584,0.811187,0.834356,0.935484,0.743017,0.975089,0.441718,0.960630
2,0.466900,0.426602,0.989162,0.989264,0.980695,0.996255,0.992701,0.987654,0.988506
3,0.418600,0.414845,0.993815,0.993865,0.992126,0.992481,0.996364,0.991935,0.996169
4,0.416300,0.403252,0.998483,0.998466,0.996078,1.000000,0.996337,1.000000,1.000000
5,0.396100,0.413715,0.992166,0.992331,0.980695,0.992481,1.000000,0.987654,1.000000
6,0.398000,0.402729,0.996907,0.996933,0.996047,0.992537,1.000000,0.995951,1.000000
7,0.395200,0.397926,0.998466,0.998466,0.996047,0.996283,1.000000,1.000000,1.000000
8,0.395000,0.398206,0.998466,0.998466,0.996047,0.996283,1.000000,1.000000,1.000000


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

TrainOutput(global_step=1208, training_loss=0.53744117471556, metrics={'train_runtime': 184.0496, 'train_samples_per_second': 523.337, 'train_steps_per_second': 16.409, 'total_flos': 278490880659552.0, 'train_loss': 0.53744117471556, 'epoch': 8.0})

## Đánh giá test set — confusion + per-length (xác nhận hết length bias)

In [9]:
import numpy as np
pred = trainer.predict(ds_test)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
print(classification_report(y_true, y_pred, target_names=INTENT_LABELS, digits=4))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
print("confusion (row=true, col=pred):")
print("      " + " ".join(f"{l[:6]:>7}" for l in INTENT_LABELS))
for i, l in enumerate(INTENT_LABELS):
    print(f"{l[:6]:>6} " + " ".join(f"{cm[i][j]:>7}" for j in range(len(INTENT_LABELS))))

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


                  precision    recall  f1-score   support

   similar_items     1.0000    1.0000    1.0000       200
   graph_pairing     0.9754    1.0000    0.9875       198
   color_variant     0.9849    1.0000    0.9924       196
composite_intent     1.0000    0.9581    0.9786       191
       chit_chat     1.0000    1.0000    1.0000       189

        accuracy                         0.9918       974
       macro avg     0.9921    0.9916    0.9917       974
    weighted avg     0.9920    0.9918    0.9917       974

confusion (row=true, col=pred):
       simila  graph_  color_  compos  chit_c
simila     200       0       0       0       0
graph_       0     198       0       0       0
color_       0       0     196       0       0
compos       0       5       3     183       0
chit_c       0       0       0       0     189


In [10]:
test_eval = test_df.copy().reset_index(drop=True)
test_eval["pred"] = [ID2LABEL[i] for i in y_pred]
test_eval["n_words"] = test_eval["query"].astype(str).str.split().str.len()
test_eval["len_bucket"] = pd.cut(test_eval["n_words"], [0, 4, 12, 100], labels=["short", "medium", "long"])
for b in ["short", "medium", "long"]:
    sub = test_eval[test_eval["len_bucket"] == b]
    if len(sub):
        print(f"{b:7s} n={len(sub):4d} acc={(sub['pred']==sub['intent']).mean():.4f}")
print("long-query per intent:")
longsub = test_eval[test_eval["len_bucket"] == "long"]
for intent in INTENT_LABELS:
    s = longsub[longsub["intent"] == intent]
    if len(s):
        print(f"  {intent:18s} n={len(s):3d} acc={(s['pred']==s['intent']).mean():.3f}")

short   n= 222 acc=0.2793
medium  n= 403 acc=0.1935
long    n= 349 acc=0.2206
long-query per intent:
  similar_items      n= 70 acc=0.300
  graph_pairing      n= 70 acc=0.271
  color_variant      n= 69 acc=0.188
  composite_intent   n= 70 acc=0.100
  chit_chat          n= 70 acc=0.243


In [11]:
REAL_CASES = [
    ("I want to find a sleek, greenish khaki padded jacket. It features a zip and wind flap with press-studs", "similar_items"),
    ("sun-faded charcoal 14 oz box hoodie with cropped hem and oversized fit in washed black", "similar_items"),
    ("a flowy bohemian maxi dress in dusty rose with delicate floral embroidery and bell sleeves for summer", "similar_items"),
    ("padded jacket", "similar_items"),
    ("find a bag that matches this dress", "graph_pairing"),
    ("what shoes would go well with these wide-leg linen trousers", "graph_pairing"),
    ("is this available in red", "color_variant"),
    ("does this hoodie come in other colors", "color_variant"),
    ("find me a white dress and earrings to go with it", "composite_intent"),
    ("show this in black and also suggest a belt to pair", "composite_intent"),
    ("hello how are you", "chit_chat"),
    ("where is my order", "chit_chat"),
]
enc = tokenizer([q for q, _ in REAL_CASES], return_tensors="pt", truncation=True, max_length=MAX_LENGTH, padding=True).to(model.device)
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1)
correct = 0
for i, (q, exp) in enumerate(REAL_CASES):
    idx = int(probs[i].argmax())
    pred = ID2LABEL[idx]
    correct += pred == exp
    mark = "ok" if pred == exp else "X"
    print(f"{mark:<3}{pred:<18}{exp:<18}{float(probs[i][idx]):.2f}  {q[:50]}")
print(f"REAL-CASE accuracy: {correct}/{len(REAL_CASES)} = {correct/len(REAL_CASES):.2f}")

ok similar_items     similar_items     0.95  I want to find a sleek, greenish khaki padded jack
ok similar_items     similar_items     0.95  sun-faded charcoal 14 oz box hoodie with cropped h
ok similar_items     similar_items     0.94  a flowy bohemian maxi dress in dusty rose with del
ok similar_items     similar_items     0.95  padded jacket
ok graph_pairing     graph_pairing     0.95  find a bag that matches this dress
ok graph_pairing     graph_pairing     0.95  what shoes would go well with these wide-leg linen
ok color_variant     color_variant     0.94  is this available in red
ok color_variant     color_variant     0.94  does this hoodie come in other colors
ok composite_intent  composite_intent  0.95  find me a white dress and earrings to go with it
ok composite_intent  composite_intent  0.94  show this in black and also suggest a belt to pair
ok chit_chat         chit_chat         0.95  hello how are you
ok chit_chat         chit_chat         0.95  where is my order
REAL-CAS

In [13]:
import json
from sklearn.metrics import f1_score, accuracy_score
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/intent_classifier_deberta"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/label_map.json", "w") as f:
    json.dump(LABEL2ID, f, indent=2)

metrics = {
    "backbone": BACKBONE,
    "max_length": MAX_LENGTH,
    "test_f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_f1_per_class": {INTENT_LABELS[i]: float(v) for i, v in
                          enumerate(f1_score(y_true, y_pred, average=None, labels=list(range(len(INTENT_LABELS)))))},
    "config": {"lr": LR, "batch": BATCH, "label_smoothing": LABEL_SMOOTHING,
               "llrd_decay": LLRD_DECAY, "scheduler": LR_SCHEDULER, "seed": SEED},
}
with open(f"{SAVE_DIR}/eval_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("saved ->", SAVE_DIR)


Mounted at /content/drive
{
  "backbone": "microsoft/deberta-v3-base",
  "max_length": 128,
  "test_f1_macro": 0.9917091722058828,
  "test_accuracy": 0.9917864476386037,
  "test_f1_per_class": {
    "similar_items": 1.0,
    "graph_pairing": 0.9875311720698254,
    "color_variant": 0.9924050632911392,
    "composite_intent": 0.9786096256684492,
    "chit_chat": 1.0
  },
  "config": {
    "lr": 2e-05,
    "batch": 32,
    "label_smoothing": 0.1,
    "llrd_decay": 0.9,
    "scheduler": "cosine",
    "seed": 42
  }
}
saved -> /content/drive/MyDrive/intent_classifier_deberta
